In [ ]:
import os
import torch

print("GPU VERIFICATION:")
num_gpus_total = torch.cuda.device_count() if torch.cuda.is_available() else 0
usable_gpus = []
for idx in range(num_gpus_total):
    major, minor = torch.cuda.get_device_capability(idx)
    name = torch.cuda.get_device_name(idx)
    ok = major >= 6
    if ok:
        usable_gpus.append(idx)
    print(f"  GPU {idx}: {name} (compute capability {major}.{minor}) -> {'usable' if ok else 'SKIPPED (too old for AMP)'}")

device = torch.device(f"cuda:{usable_gpus[0]}" if usable_gpus else "cpu")
if usable_gpus:
    print(f"\nSUCCESS: {len(usable_gpus)} usable GPU(s) detected. pipeline.py will use {device} as primary"
          f"{' and DataParallel across all usable GPUs during training' if len(usable_gpus) > 1 else ''}.")
else:
    print("\nWARNING: No compatible GPU detected! Pipeline will run on CPU.")
print()

repo_dir = "/kaggle/working/zerocross-ai"
if not os.path.exists(repo_dir):
    !git clone https://github.com/muhammad-hassaan-aiml/zerocross-ai.git {repo_dir}
else:
    # /kaggle/working persists across commits within the same session, so a repo
    # cloned earlier in this session can just be updated in place instead of
    # silently re-using whatever code happened to be cloned last time.
    print(f"{repo_dir} already exists -- pulling latest changes instead of re-cloning...")
    !git -C {repo_dir} pull --ff-only

os.chdir(repo_dir)
!chmod +x build_kaggle.sh
!./build_kaggle.sh


In [ ]:
import os
import shutil

input_dir = "/kaggle/input/"
working_models_dir = "/kaggle/working/models"
os.makedirs(working_models_dir, exist_ok=True)

found_previous = False
if os.path.exists(input_dir):
    for root, dirs, files in os.walk(input_dir):
        if "best_model.pth" in files:
            print(f"Found previous session data in {root}!")
            print("Copying to working directory to resume training...")
            for file in files:
                if file.endswith((".pth", ".pt", ".csv", ".json")):
                    shutil.copy(os.path.join(root, file), working_models_dir)
            found_previous = True
            break

if not found_previous:
    print("No previous dataset attached. Starting a fresh session.")
else:
    print("Resume data loaded successfully!")


In [ ]:
import json, os, math, torch

state_path = "/kaggle/working/models/pipeline_state.json"
model_path = "/kaggle/working/models/best_model.pth"

# Keep this equal to --max-rejections in the training cell below -- it only
# affects this preview, pipeline.py tracks and applies the real thing itself.
MAX_REJECTIONS = 5

total_iterations = 0
consecutive_rejections = 0
if os.path.exists(state_path):
    state_data = json.load(open(state_path))
    total_iterations = state_data.get("total_iterations", 0)
    consecutive_rejections = state_data.get("consecutive_rejections", 0)

if total_iterations == 0 and os.path.exists(model_path):
    ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
    if isinstance(ckpt, dict):
        total_iterations = ckpt.get("iteration", 0)
    print("(state file missing/zero -- falling back to checkpoint's embedded iteration)")

next_iter = total_iterations + 1
if next_iter <= 100:
    lr = 0.001
elif next_iter <= 250:
    lr = 0.0005
else:
    lr = 0.0001

print(f"total_iterations (resume point):  {total_iterations}")
print(f"next iteration will be:           {next_iter}")
print(f"learning rate that implies:       {lr}")
print(f"consecutive rejections:           {consecutive_rejections} / {MAX_REJECTIONS}")

stall_boost_at = max(1, math.ceil(MAX_REJECTIONS / 2))
if consecutive_rejections >= MAX_REJECTIONS:
    print("  -> at the forced-promotion threshold: the next rejection only forces")
    print("     a promotion through if win rate vs champion clears --min-force-promote-winrate")
elif consecutive_rejections >= stall_boost_at:
    print(f"  -> past the stall-boost threshold ({stall_boost_at}): evaluations are being")
    print("     widened automatically to cut through noise before any forced promotion")
print()


In [ ]:
!python python/pipeline.py \
    --iterations 100 \
    --concurrent-games 400 \
    --games-per-iteration 500 \
    --mcts-sims 200 \
    --eval-games 100 \
    --eval-sims 200 \
    --batch-size 2048 \
    --max-rejections 5 \
    --min-force-promote-winrate 0.45 \
    --stall-eval-multiplier 3 \
    --num-res-blocks 6 \
    --num-channels 128 \
    --max-buffer-size 1000000 \
    --buffer-archive-interval 5


In [ ]:
import json
print(json.dumps(json.load(open("/kaggle/working/models/pipeline_state.json")), indent=2))

In [ ]:
import sys, glob
sys.path.append("/kaggle/working/zerocross-ai/python")
from plot_metrics import plot_training_metrics
from IPython.display import Image, display

plot_training_metrics()
for img_path in sorted(glob.glob("/kaggle/working/models/plots/*.png")):
    display(Image(filename=img_path))